In [ ]:
# ==============================================================================
# IDENTIFICAÇÃO DO GRUPO:
# - Rafael Moutinho Tessarotto (RA: 10395682)
# - Edson Fu (RA: 10419137 )
# - Rafael Santos Lourenço da Silva (RA: 10403588 )
# 
# SÍNTESE DO CONTEÚDO:
# Este notebook implementa um sistema inteligente de pré-triagem médica baseado
# na Opção LLM da disciplina. O código carrega datasets de referência, realiza o 
# mapeamento de sintomas em categorias estatísticas e utiliza o Llama 3 (via Ollama) 
# para estruturar queixas em linguagem leiga para o formato de anamnese (JSON). 
# O sistema atua de forma estritamente organizacional e possui travas éticas 
# para não fornecer diagnósticos ou recomendações clínicas.
#
# HISTÓRICO DE ALTERAÇÕES:
# - [Data Anterior]: [Autor] - Criação da lógica de mapeamento de categorias e estatísticas.
# - [Data Anterior]: [Autor] - Implementação da integração com Ollama e few-shot prompting.
# - 28/05/2026: Rafael Moutinho Tessarotto (e equipe) - Inclusão do cabeçalho obrigatório, 
#               revisão final dos aspectos éticos e preparação para a submissão da N2.
# ==============================================================================

In [ ]:
import json
import pandas as pd
import ollama

In [ ]:
# Datasets:

# 1. dataset_triagem.csv
#     Contém exemplos clínicos utilizados
#     como referência para o modelo.
# 2. estatisticas_por_categoria.csv
#     Contém estatísticas históricas
#     de categorias relacionadas aos sintomas.
# 3. durchlauf1.csv
#     Dataset complementar contendo
#     registros históricos adicionais.



In [ ]:
dataset_triagem = pd.read_csv(
"dataset_triagem.csv"
)

estatisticas = pd.read_csv(
"estatisticas_por_categoria.csv"
)

durchlauf = pd.read_csv(
"durchlauf1.csv",
sep=";"
)

print("=" * 60)
print("DATASETS CARREGADOS")
print("=" * 60)

print("\nDataset triagem:")
print(dataset_triagem.shape)

print("\nEstatísticas:")
print(estatisticas.shape)

print("\nDurchlauf:")
print(durchlauf.shape)

In [ ]:
# MAPA DE CATEGORIAS ESTATISTICAS
    
#     Relacionar sintomas identificados
#     com categorias estatísticas históricas.


In [ ]:
MAPA_CATEGORIAS = {

"dor torácica": 2,
"dispneia": 2,
"falta de ar": 2,

"febre": 4,
"cefaleia": 4,
"tosse": 4,

"êmese": 3,
"vomito": 3,
"vômito": 3

}

In [ ]:
# IDENTIFICAÇÃO DE CATEGORIA

# OBJETIVO:
# Identificar uma categoria estatística
# relacionada aos sintomas encontrados.

# A função:

# recebe o texto da anamnese;
# procura sintomas conhecidos;
# retorna uma categoria histórica relacionada.

# Caso nenhum sintoma seja encontrado,
# a categoria padrão retornada será 5.


In [ ]:
def identificar_categoria(texto):

    texto = texto.lower()

    for sintoma, categoria in MAPA_CATEGORIAS.items():

        if sintoma in texto:
            return categoria

    return 5

In [ ]:
# CONSULTA ESTATÍSTICA 
#     OBJETIVO:
#     Buscar estatísticas históricas
#     relacionadas à categoria identificada.

#     As estatísticas incluem:

#         tempo médio;
#         tempo mediano;
#         tempo mínimo;
#         tempo máximo.

#     Esses dados são apenas informativos
#     e NÃO representam recomendação médica.


In [ ]:
def buscar_estatisticas(categoria):

    linha = estatisticas[
        estatisticas["triagestufe"] == categoria
    ]

    if len(linha) == 0:
        return None

    linha = linha.iloc[0]

    return {

        "tempo_medio":
            linha["mean"],

        "tempo_mediano":
            linha["median"],

        "tempo_minimo":
            linha["min"],

        "tempo_maximo":
            linha["max"]
    }

In [ ]:
# OBJETIVO:
#     Construir o prompt enviado ao modelo LLM.

#     O prompt contém:

#         instruções éticas;
#         regras de comportamento;
#         exemplos do dataset;
#         caso atual do paciente.

#     Os exemplos presentes no dataset
#     são utilizados como few-shot prompting,
#     permitindo que o modelo imite
#     o formato das anamneses fornecidas.

In [ ]:
def construir_prompt(texto_paciente):

    prompt = """

    Você é um assistente inteligente
    de pré-triagem médica.

    Sua função é:

    organizar sintomas;
    estruturar informações clínicas;
    converter linguagem leiga em termos médicos;
    gerar uma anamnese padronizada.

    REGRAS OBRIGATÓRIAS:

    Nunca realize diagnóstico.
    Nunca sugira tratamento.
    Nunca afirme doenças.
    Nunca classifique risco clínico.
    Nunca substitua avaliação médica.

    Você deve apenas organizar informações.

    Responda APENAS em JSON válido.

    Formato esperado:

    {
    "queixa_principal": "",
    "duracao": "",
    "intensidade": "",
    "sintomas_associados": [],
    "historico_medico": [],
    "anamnese": ""
    }

    EXEMPLOS:
    """

    for _, linha in dataset_triagem.iterrows():

        sintomas_associados = []
        historico_medico = []

        if pd.notna(
            linha["sintomas_associados"]
        ):

            sintomas_associados = [

                item.strip()

                for item in str(
                    linha["sintomas_associados"]
                ).split(";")

                if item.strip()
            ]

        if pd.notna(
            linha["historico_medico"]
        ):

            historico_medico = [

                item.strip()

                for item in str(
                    linha["historico_medico"]
                ).split(";")

                if item.strip()
            ]

        saida = {

            "queixa_principal":
                linha["queixa_principal"],

            "duracao":
                linha["duracao"],

            "intensidade":
                linha["intensidade"],

            "sintomas_associados":
                sintomas_associados,

            "historico_medico":
                historico_medico,

            "anamnese":
                linha["anamnese"]
        }

        prompt += f"""

    ENTRADA:
    {linha["entrada"]}

    SAÍDA:
    {json.dumps(saida, ensure_ascii=False, indent=4)}
    """

    prompt += f"""

    Agora analise o seguinte caso:

    ENTRADA:
    {texto_paciente}

    SAÍDA:
    """

    return prompt

In [ ]:
# OBJETIVO:
# Enviar o prompt ao modelo LLM
# e receber a resposta estruturada.

# O modelo utilizado é o Llama 3,
# executado localmente via Ollama.

In [ ]:
def gerar_anamnese(texto_paciente):

    prompt = construir_prompt(
        texto_paciente
    )

    resposta = ollama.chat(

        model="llama3",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return resposta["message"]["content"]

In [ ]:
# OBJETIVO:
# Receber os sintomas do paciente
# via linha de comando.

# Também exibe avisos éticos
# antes da interação com o usuário.

In [ ]:
print("\n")
print("=" * 60)
print("SISTEMA INTELIGENTE DE PRÉ-TRIAGEM")
print("=" * 60)

print("""
AVISO IMPORTANTE

Este sistema NÃO realiza diagnóstico,
NÃO substitui profissionais de saúde
e NÃO realiza classificação clínica.

O sistema apenas organiza informações
para apoio administrativo.
""")

entrada = input(
"\nDescreva seus sintomas:\n> "
)

resultado = gerar_anamnese(entrada)

print("\n")
print("=" * 60)
print("RESPOSTA DO MODELO")
print("=" * 60)

print(resultado)

In [ ]:
# OBJETIVO:
# Converter a resposta JSON gerada pelo modelo
# em uma estrutura organizada e legível.

# Nesta etapa o sistema:

# exibe a anamnese estruturada;
# identifica uma categoria estatística relacionada;
# apresenta estatísticas históricas associadas.

# IMPORTANTE:
# As estatísticas apresentadas possuem
# apenas finalidade informativa.

In [ ]:
try:

    dados = json.loads(resultado)

    texto_busca = (

        dados["queixa_principal"]

        + " "

        + " ".join(
            dados[
                "sintomas_associados"
            ]
        )
    )

    categoria = identificar_categoria(
        texto_busca
    )

    estatisticas_categoria = (
        buscar_estatisticas(
            categoria
        )
    )

    print("\n")
    print("=" * 60)
    print("ANAMNESE ESTRUTURADA")
    print("=" * 60)

    print(f"\nQueixa principal:")
    print(dados["queixa_principal"])

    print(f"\nDuração:")
    print(dados["duracao"])

    print(f"\nIntensidade:")
    print(dados["intensidade"])

    print(f"\nSintomas associados:")

    if len(
        dados["sintomas_associados"]
    ) == 0:

        print("- Nenhum")

    else:

        for sintoma in dados[
            "sintomas_associados"
        ]:

            print(f"- {sintoma}")

    print(f"\nHistórico médico:")

    if len(
        dados["historico_medico"]
    ) == 0:

        print("- Nenhum")

    else:

        for item in dados[
            "historico_medico"
        ]:

            print(f"- {item}")

    print(f"\nAnamnese:")
    print(dados["anamnese"])

    print("\n")
    print("=" * 60)
    print("DADOS ESTATÍSTICOS RELACIONADOS")
    print("=" * 60)

    print("""

    Os dados abaixo representam apenas
    estatísticas históricas relacionadas
    a sintomas semelhantes.

    Eles NÃO representam:

    diagnóstico;
    classificação médica;

    decisão clínica.
    """)

    print(f"\nCategoria estatística:")
    print(categoria)

    if estatisticas_categoria is not None:

        print(f"\nTempo médio histórico:")
        print(
            estatisticas_categoria[
                "tempo_medio"
            ]
        )

        print(f"\nTempo mediano histórico:")
        print(
            estatisticas_categoria[
                "tempo_mediano"
            ]
        )

        print(f"\nTempo mínimo histórico:")
        print(
            estatisticas_categoria[
                "tempo_minimo"
            ]
        )

        print(f"\nTempo máximo histórico:")
        print(
            estatisticas_categoria[
                "tempo_maximo"
            ]
        )
except Exception as erro:
    print("\nErro ao processar resposta:")
    print(erro)